<div class="alert alert-primary" style="margin-top: 20px">


<h1><center><Strong>IMDB Sentiment Analysis</strong></center></h1>

</div>


<img algin="center" src="https://upload.wikimedia.org/wikipedia/commons/6/69/IMDB_Logo_2016.svg"/>

<div class="alert alert-primary" style="margin-top: 20px">

<strong><h2>Dataset Story:</h2></strong>
<blockquote><p>IMDB dataset having 50K movie reviews for natural language processing or Text analytics.
This is a dataset for binary sentiment classification containing substantially more data than previous benchmark datasets. We provide a set of 25,000 highly polar movie reviews for training and 25,000 for testing. So, predict the number of positive and negative reviews using either classification or deep learning algorithms.</p>
</blockquote>
<ul>
    
</ul>
<strong><h2>Features:</h2></strong>
<ul>
<li>Total Features : 2</li>
<li>Total Row : 50000 </li>
<li>CSV File Size : 66.21 MB</li>

</ul>
</div>

 Sr. | Feature  | Description |
--- | --- | --- 
1 | Review | User comment about the movie
2 | Sentiment | The sentiment of the comment made about the movie {Positive / Negative }

---

### [For more dataset information](http://ai.stanford.edu/~amaas/data/sentiment/)    

 




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from bs4 import BeautifulSoup
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from nltk.corpus import stopwords

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Loading Data
df_ = pd.read_csv('IMDB Dataset.csv')

In [ ]:
df = df_.copy()

## Examining Data
---

In [ ]:
def missing_values_analysis(df):
    na_columns_ = [col for col in df.columns if df[col].isnull().sum() > 0]
    n_miss = df[na_columns_].isnull().sum().sort_values(ascending=True)
    ratio_ = (df[na_columns_].isnull().sum() / df.shape[0] * 100).sort_values(ascending=True)
    missing_df = pd.concat([n_miss, np.round(ratio_, 2)], axis=1, keys=['Total Missing Values', 'Ratio'])
    missing_df = pd.DataFrame(missing_df)
    return missing_df

def check_df(df, head=5):
    print("--------------------- Shape --------------------")
    print(df.shape)
    print("-------------------- Types ---------------------")
    print(df.dtypes)
    print("----------------- NaN Analysis -----------------")
    print(missing_values_analysis(df))
    print("--------------------- Head ---------------------")
    print(df.head())

check_df(df)

In [ ]:
a , ax = plt.subplots(1,2,figsize=(22,9))
df['sentiment'].value_counts().plot.pie(explode=[0,0.1], autopct='%1.2f%%',ax=ax[0],shadow=True, startangle=300, colors = ["#bcbddc", "#efedf5"])
ax[0].set_title('Distribution of Positive / Negative Emotions')
ax[0].set_ylabel('')
sns.countplot('sentiment', data = df ,ax=ax[1], palette=["#bcbddc", "#efedf5"])
ax[1].set_title('Distribution of Positive / Negative Emotions')
plt.show()

In [ ]:
df.groupby(['sentiment'])[['sentiment']].count()

## Everything is going well...
----

<div class="alert alert-block alert-info">
        <b> ??We didn't find any missing values and everything seems fine. The distribution of our data is good. There are 50,000 observation units in total (Emotional distribution: positive 50% / negative 50%) and 2 features. We can move on to the cleaning phase.</b>
</div>
<br>

In [ ]:
# First, we will delete HTML tags from review sentences using the BeautifulSoup module, 

<div style="color:white;
           display:fill;
           border-radius:5px;
           background-color:crimson;
           font-size:110%;
           font-family:Verdana;
           letter-spacing:0.5px">
?? To explain how these processes are done, first let's choose a single review and see how it is done for you:
</div>

In [ ]:
example_review = df.review[0]
example_review

In [ ]:
# After cleaning the HTML tags... ??
example_review = BeautifulSoup(example_review).get_text()
example_review

In [ ]:
# we clean it from punctuation and numbers - using regex. {Regular expression}
example_review = re.sub("[^a-zA-Z]",' ',example_review)
example_review

## An important point! - 1
---

<div class="alert alert-block alert-info">
        <b> ??I will now convert the text to all lowercase.
Our machine learning algorithms recognize words that start with a capital letter as different words, and we will convert them to lowercase. Thus, our machine learning algorithms will not perceive words that start with a capital letter as a different word.</b>
</div>
<br>

In [ ]:
example_review = example_review.lower()
example_review

## An important point! - 2
---
<div class="alert alert-block alert-info">
        <b> ??Stopwords (We want words like the, is, are not to be used by AI. These are grammar words..) That's why we're going to remove them. </b>
</div>
<br>

In [ ]:
example_review = example_review.split()

In [ ]:
nltk.download('stopwords')

In [ ]:
words = set(stopwords.words("english")) # conversion into set for fast searching !!!
example_review = [word for word in example_review if word not in words]              
example_review

<div style="color:white;
           display:fill;
           border-radius:5px;
           background-color:blue;
           font-size:110%;
           font-family:Verdana;
           letter-spacing:0.5px">
    <p> ?? Perfect if you understand this cleaning process (through the example.) ?????</p>
</div>

## Now let's apply this cleanup to the entire dataframe
---

In [ ]:
def process(review):
    # review without HTML tags
    review = BeautifulSoup(review).get_text()
    # review without punctuation and numbers
    review = re.sub("[^a-zA-Z]",' ',review)
    # converting into lowercase and splitting to eliminate stopwords
    review = review.lower()
    review = review.split()
    # review without stopwords
    swords = set(stopwords.words("english"))  # conversion into set for fast searching
    review = [w for w in review if w not in swords]               
    # splitted paragraph'lar? space ile birle?tiriyoruz return
    return(" ".join(review))

In [ ]:
# We can see the status of the review process by printing a line after every 2500 reviews.

train_data = []
for i in range(len(df["review"])):        
    if (i+1)%2500 == 0:        
        print("Result =", i+1)
    train_data.append(process(df["review"][i]))

## Split : Train Path / Test Path 
---

In [ ]:
independent = train_data
dependent = np.array(df["sentiment"])

train_x, test_x, y_train, y_test = train_test_split(independent,dependent, stratify = dependent ,test_size = 0.2, shuffle = True , random_state = 25)


### We are creating Bag of Words!
---
<div class="alert alert-block alert-info">
        <b> ??We have cleaned our data, but for the artificial intelligence to work, it is necessary to convert this text-based data into numbers and a matrix called bag of words. Here we use the CountVectorizer tool in sklearn for this purpose ????</b>
</div>
<br>

<img src="https://teknofesor.com/wp-content/uploads/2020/01/bag-of-words-nedir.png" width="900" height="900" >

In [ ]:
# # Using the countvectorizer function in sklearn, we create a bag of words with a maximum of 10.000 words...
vectorizer = CountVectorizer( max_features = 10000 )

# we convert our train data to feature vector matrix
train_x = vectorizer.fit_transform(train_x)

In [ ]:
vectorizer

In [ ]:
train_x

In [ ]:
# Since our machine learning algorithm requires an array for the training process, we convert it to an array.
train_x = train_x.toarray()
train_y = y_train

In [ ]:
print("Total training data :", train_x.shape[0])
print("Total training data :", train_y.shape[0])

# Model evaluation

In [ ]:
models = []

models.append(('Logistic Regression',LogisticRegression()))
models.append(('Random Forest Classifier',RandomForestClassifier()))
models.append(('Decision Tree Classifier',DecisionTreeClassifier()))

In [ ]:
# We convert our test data to vector matrix feature
#This time for our test:
test_result = vectorizer.transform(test_x)


In [ ]:
test_result = test_result.toarray()

In [ ]:
test_result.shape

In [ ]:
for name, model in models:

    %time model.fit(train_x, train_y)
    test_pred = model.predict(test_result)
    print(name ,'Accuracy Score : ',accuracy_score(y_test, test_pred))
    print(name ,'F1 Score : ',f1_score(y_test, test_pred, average='weighted'))
    print('-----------------------------------------------------------------------')